In [71]:
# !pip install opencv-python
# !pip install mediapipe

In [72]:
import cv2
import mediapipe as mp
import numpy as np
import os
import pandas as pd

In [73]:
mp_hands = mp.solutions.hands
hand_tracker = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def compute_motion(prev_coordinates, curr_coordinates):
    motion_vector = []
    for prev, curr in zip(prev_coordinates, curr_coordinates):
        dx = curr[0] - prev[0]
        dy = curr[1] - prev[1]
        dz = curr[2] - prev[2]
        motion_vector.append([dx, dy, dz])
    return np.array(motion_vector)

video_folder = "/Users/aksanurkonuk/PycharmProjects/Two-Way-Real-Time-Sign-Language-Converter/dataset"
motion_sequences = []
labels = []
sequence_length = 90
label_path = '/Users/aksanurkonuk/PycharmProjects/Two-Way-Real-Time-Sign-Language-Converter/dataset/labels.csv'
#df_labels = pd.read_csv(label_path) 
df_labels = {
    '0004': 'açıklamak',
    '0009': 'adres',
    '0016': 'akraba'
}

for video_file in os.listdir(video_folder):
    if video_file.endswith(".mp4"):
        video_path = os.path.join(video_folder, video_file)
        
        current_id = video_file.split(".")[0]
        #current_label = df_labels.loc[current_id, 'meaning']
        current_label = df_labels.get(current_id, 'Label not found')
        
        cap = cv2.VideoCapture(video_path)

        prev_coordinates = None
        motion_sequence = []


        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hand_tracker.process(rgb_frame)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    curr_coordinates = [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]

                    if prev_coordinates is not None:
                        motion_vector = compute_motion(prev_coordinates, curr_coordinates)
                        motion_sequence.append(motion_vector)

                        if len(motion_sequence) == sequence_length:
                            motion_sequences.append(motion_sequence)
                            labels.append(current_label)
                            motion_sequence = []

                    prev_coordinates = curr_coordinates

        cap.release()

motion_sequences = np.array(motion_sequences)
labels = np.array(labels)

np.save("motion_sequences.npy", motion_sequences)
np.save("labels.npy", labels)


I0000 00:00:1734119694.686905 1093873 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1734119694.710997 1218491 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1734119694.752220 1218491 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [74]:
import numpy as np

# Verileri yükleyin
motion_sequences = np.load('motion_sequences.npy')
labels = np.load('labels.npy')

# Verilerin şekline bakın
print(motion_sequences.shape)
print(labels.shape)


(3, 90, 21, 3)
(3,)


In [75]:
# !pip install tensorflow
from sklearn.preprocessing import LabelEncoder

# Etiketleri sayısal hale getirme
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

# One-hot encoding (isteğe bağlı)
from keras.utils import to_categorical
labels_one_hot = to_categorical(labels_encoded)


In [76]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# motion_sequences verisini (None, 90, 21, 3) şeklinde, (None, 90, 63)'e dönüştürme
motion_sequences_reshaped = motion_sequences.reshape(motion_sequences.shape[0], motion_sequences.shape[1], -1)

# Modeli oluşturun
model = Sequential()

# LSTM katmanı ekleyin (input_shape, örneğin (sequence_length, feature_count))
model.add(LSTM(64, input_shape=(motion_sequences_reshaped.shape[1], motion_sequences_reshaped.shape[2]), return_sequences=False))
model.add(Dropout(0.2))

# Çıkış katmanı (etkili sınıf sayısı kadar nöron)
model.add(Dense(labels_one_hot.shape[1], activation='softmax'))

# Modeli derleyin
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Modelin özetini görüntüleyin
model.summary()

# Modeli eğitme
history = model.fit(motion_sequences_reshaped, labels_one_hot, epochs=50, batch_size=32, validation_split=0.2)

# Modeli kaydetme
model.save('sign_language_model.h5')


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 64)             │        32,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,963 (128.76 KB)

 Trainable params: 32,963 (128.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 683ms/step - accuracy: 0.5000 - loss: 1.0960 - val_accuracy: 0.0000e+00 - val_loss: 1.1050
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5000 - loss: 1.0865 - val_accuracy: 0.0000e+00 - val_loss: 1.1127
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.5000 - loss: 1.0857 - val_accuracy: 0.0000e+00 - val_loss: 1.1212
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 1.0000 - loss: 1.0749 - val_accuracy: 0.0000e+00 - val_loss: 1.1306
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 1.0644 - val_accuracy: 0.0000e+00 - val_loss: 1.1410
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 1.0572 - val_accuracy: 0.0000e+00 - val_loss: 1.1524
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 1.0527 - val_accuracy: 0.0000e+00 - val_loss: 1.1654
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 1.0400 - val_accurac

In [77]:
# Modeli eğitme
#history = model.fit(motion_sequences, labels_one_hot, epochs=50, batch_size=32, validation_split=0.2)


In [78]:
# Modeli kaydet
#model.save('sign_language_model.h5')
